# YOLO Object Detection Pipeline

Pipeline untuk berbagai use case object detection menggunakan YOLO (YOLOv5, YOLOv8, YOLOv10).

## Use Cases:
1. General Object Detection
2. Custom Object Detection
3. Instance Segmentation
4. Pose Estimation
5. Object Tracking
6. Real-time Detection
7. Batch Processing
8. Model Training
9. Model Export & Deployment

## 1. Setup & Installation

In [ ]:
import torch
import cv2
import numpy as np
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"Device: {torch.cuda.get_device_name(0)}")

## 2. Load YOLO Models

In [ ]:
from ultralytics import YOLO

def load_yolo_model(model_name='yolov8n.pt', device='cuda'):
    """Load YOLO model"""
    model = YOLO(model_name)
    model.to(device)
    return model

# Available models:
# YOLOv8: yolov8n.pt, yolov8s.pt, yolov8m.pt, yolov8l.pt, yolov8x.pt
# YOLOv8-seg: yolov8n-seg.pt, yolov8s-seg.pt, yolov8m-seg.pt
# YOLOv8-pose: yolov8n-pose.pt, yolov8s-pose.pt
# YOLOv10: yolov10n.pt, yolov10s.pt, yolov10m.pt, yolov10l.pt, yolov10x.pt

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = load_yolo_model('yolov8n.pt', device)
print(f"Model loaded on {device}")

## 3. Pipeline 1: Single Image Detection

In [ ]:
def detect_single_image(model, image_path, conf=0.25, iou=0.45):
    """Detect objects in single image"""
    results = model.predict(
        source=image_path,
        conf=conf,
        iou=iou,
        device=device
    )
    return results[0]

def visualize_detection(results, save_path=None):
    """Visualize detection results"""
    img = results.plot()
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    plt.figure(figsize=(12, 8))
    plt.imshow(img_rgb)
    plt.axis('off')
    
    if save_path:
        plt.savefig(save_path, bbox_inches='tight', dpi=150)
    plt.show()
    
    # Print detections
    boxes = results.boxes
    print(f"Detected {len(boxes)} objects")
    for box in boxes:
        cls = int(box.cls[0])
        conf = float(box.conf[0])
        label = results.names[cls]
        print(f"  {label}: {conf:.2f}")

# Example usage
# results = detect_single_image(model, 'image.jpg', conf=0.5)
# visualize_detection(results, save_path='output.jpg')

## 4. Pipeline 2: Batch Image Detection

In [ ]:
def detect_batch_images(model, image_dir, output_dir, conf=0.25, save_txt=True):
    """Detect objects in multiple images"""
    image_dir = Path(image_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    image_paths = list(image_dir.glob('*.jpg')) + list(image_dir.glob('*.png'))
    
    results = model.predict(
        source=image_paths,
        conf=conf,
        save=True,
        save_txt=save_txt,
        project=str(output_dir),
        name='batch_detection',
        device=device
    )
    
    print(f"Processed {len(results)} images")
    return results

# Example usage
# results = detect_batch_images(model, 'input_images/', 'output/', conf=0.5)

## 5. Pipeline 3: Video Detection

In [ ]:
def detect_video(model, video_path, output_path, conf=0.25):
    """Detect objects in video"""
    results = model.predict(
        source=video_path,
        conf=conf,
        save=True,
        project=str(Path(output_path).parent),
        name=Path(output_path).stem,
        device=device
    )
    
    print(f"Video processed: {output_path}")
    return results

# Example usage
# results = detect_video(model, 'input.mp4', 'output/result.mp4', conf=0.5)

## 6. Pipeline 4: Real-time Webcam Detection

In [ ]:
def detect_webcam(model, conf=0.25, camera_id=0):
    """Real-time detection from webcam"""
    cap = cv2.VideoCapture(camera_id)
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        
        results = model.predict(frame, conf=conf, device=device, verbose=False)
        annotated_frame = results[0].plot()
        
        cv2.imshow('YOLO Detection', annotated_frame)
        
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
    
    cap.release()
    cv2.destroyAllWindows()

# Example usage
# detect_webcam(model, conf=0.5, camera_id=0)

## 7. Pipeline 5: Object Tracking

In [ ]:
def track_objects(model, source, output_path, tracker='bytetrack.yaml'):
    """Track objects across frames"""
    results = model.track(
        source=source,
        conf=0.3,
        iou=0.5,
        tracker=tracker,
        save=True,
        project=str(Path(output_path).parent),
        name=Path(output_path).stem,
        device=device
    )
    
    print(f"Tracking completed: {output_path}")
    return results

# Available trackers: bytetrack.yaml, botsort.yaml
# Example usage
# results = track_objects(model, 'video.mp4', 'output/tracked.mp4', tracker='bytetrack.yaml')

## 8. Pipeline 6: Instance Segmentation

In [ ]:
def segment_instances(image_path, model_name='yolov8n-seg.pt', conf=0.25):
    """Perform instance segmentation"""
    seg_model = YOLO(model_name)
    seg_model.to(device)
    
    results = seg_model.predict(
        source=image_path,
        conf=conf,
        device=device
    )
    
    result = results[0]
    
    # Visualize
    img = result.plot()
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    plt.figure(figsize=(12, 8))
    plt.imshow(img_rgb)
    plt.axis('off')
    plt.show()
    
    # Get masks
    if result.masks is not None:
        masks = result.masks.data.cpu().numpy()
        print(f"Detected {len(masks)} instances")
    
    return result

# Example usage
# result = segment_instances('image.jpg', model_name='yolov8n-seg.pt', conf=0.5)

## 9. Pipeline 7: Pose Estimation

In [ ]:
def estimate_pose(image_path, model_name='yolov8n-pose.pt', conf=0.25):
    """Estimate human pose keypoints"""
    pose_model = YOLO(model_name)
    pose_model.to(device)
    
    results = pose_model.predict(
        source=image_path,
        conf=conf,
        device=device
    )
    
    result = results[0]
    
    # Visualize
    img = result.plot()
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    plt.figure(figsize=(12, 8))
    plt.imshow(img_rgb)
    plt.axis('off')
    plt.show()
    
    # Get keypoints
    if result.keypoints is not None:
        keypoints = result.keypoints.data.cpu().numpy()
        print(f"Detected {len(keypoints)} persons")
    
    return result

# Example usage
# result = estimate_pose('image.jpg', model_name='yolov8n-pose.pt', conf=0.5)

## 10. Pipeline 8: Custom Object Detection Training

In [ ]:
def train_custom_model(data_yaml, model_name='yolov8n.pt', epochs=100, imgsz=640, batch=16):
    """Train custom YOLO model"""
    model = YOLO(model_name)
    
    results = model.train(
        data=data_yaml,
        epochs=epochs,
        imgsz=imgsz,
        batch=batch,
        device=device,
        workers=8,
        patience=50,
        save=True,
        plots=True
    )
    
    print("Training completed")
    return results

# Data YAML format:
# path: /path/to/dataset
# train: images/train
# val: images/val
# names:
#   0: class1
#   1: class2

# Example usage
# results = train_custom_model('data.yaml', model_name='yolov8n.pt', epochs=100)

## 11. Pipeline 9: Model Validation

In [ ]:
def validate_model(model, data_yaml):
    """Validate model performance"""
    metrics = model.val(data=data_yaml, device=device)
    
    print("Validation Metrics:")
    print(f"mAP50: {metrics.box.map50:.4f}")
    print(f"mAP50-95: {metrics.box.map:.4f}")
    print(f"Precision: {metrics.box.mp:.4f}")
    print(f"Recall: {metrics.box.mr:.4f}")
    
    return metrics

# Example usage
# metrics = validate_model(model, 'data.yaml')

## 12. Pipeline 10: Model Export & Optimization